# Exercise 3: Working with Gensim for Word2Vec Models and Embeddings

## Introduction to Gensim and Word2Vec

Gensim is a powerful library for natural language processing (NLP) and machine learning that focuses on topic modeling, document similarity, and vector space modeling. It is widely used to learn, manipulate, and perform various operations with word embeddings. Word embeddings are dense vector representations of words, where semantically similar words are closer to each other in vector space.

In this exercise, you'll train a **Skip-gram** Word2Vec model using Gensim on a large text corpus, visualize the learned embeddings, and explore some advanced features like similarity queries and loading pretrained embeddings.

## Task 1: Training a Skip-gram Word2Vec Model

Use the provided code to train a Word2Vec model on a large text corpus. The model will be trained using the Skip-gram approach, and the key parameters include:
- `vector_size`: The dimensionality of the word vectors (default: 100).
- `window`: The maximum distance between the current and predicted word within a sentence (default: 5).
- `sg`: Skip-gram model (set to 1).
- `epochs`: The number of training iterations (default: 5).
- `batch_words`: Number of words to process per batch (default: 10,000).

Once trained, the model will be saved as `skipgram_word2vec.model`. The exercise will involve inspecting the vocabulary, using the embeddings for word similarity tasks, and exploring other capabilities.

### Steps:
1. **Train the Skip-gram Word2Vec model** using the `train_skipgram_on_large_corpus()` function provided.
2. **Print the vocabulary** of the trained model.
3. **Learn word vectors**: Find the vector representation of specific words.
4. **Query most similar words**: Use the model to find the most similar words to a given word.
5. **Find most similar sentences**: Compare sentences by finding the most similar ones in terms of word vectors.

## Task 2: Visualizing Word Embeddings in 3D using TensorBoard

Once you've trained your Word2Vec model, visualize the learned word vectors in 3D using **TensorBoard**. The goal is to reduce the dimensionality of the word embeddings to 3D (using techniques like PCA) and plot them on a TensorBoard graph to inspect how the words cluster based on their semantic similarity.

### Steps:
1. Reduce the word embeddings to 3D using PCA.
2. Log the embeddings to TensorBoard.
3. Launch TensorBoard to inspect the word embeddings in 3D space.

## Task 3: Loading Pretrained Word Embeddings from Facebook

Gensim allows you to load pretrained word embeddings. In this task, you'll load the **FastText embeddings** pretrained by Facebook for both English and Arabic. These embeddings have been trained on a large corpus of text and can be used to perform word similarity, analogy tasks, and more. Gensim provides easy access to these pretrained embeddings.

### Steps:
1. Download the pretrained FastText word embeddings from Facebook (english and Arabic 300 size vectors).
Link: https://fasttext.cc/docs/en/crawl-vectors.html
2. Load the embeddings using Gensim’s `KeyedVectors` class.
3. Query the pretrained embeddings for the most similar words to a given word.



In [1]:
# Install necessary libraries if not already installed
!pip install gensim -q

In [2]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import multiprocessing

def train_skipgram_on_large_corpus(corpus_path, vector_size=100, window=5, sg=1, epochs=5, batch_words=10000):
    """
    Trains a skip-gram Word2Vec model on a large text corpus using Gensim, optimized for GPU if available.

    Args:
        corpus_path (str): Path to the text corpus file (e.g., a .txt file with large text data).
        vector_size (int): Dimensionality of the word vectors (default: 100).
        window (int): Maximum distance between the current and predicted word within a sentence (default: 5).
        sg (int): Defines the training algorithm. 1 for skip-gram; 0 for CBOW (default: 1, skip-gram).
        epochs (int): Number of training iterations over the entire corpus (default: 5).
        batch_words (int): Number of words to process per batch (default: 10,000).

    Returns:
        model (gensim.models.Word2Vec): Trained Word2Vec model containing word embeddings.
    """
    import os
    from gensim.models.word2vec import LineSentence

    # Ensure the corpus file exists
    if not os.path.exists(corpus_path):
        raise FileNotFoundError(f"Corpus file '{corpus_path}' not found!")

    # Load the corpus using LineSentence for memory efficiency
    # LineSentence processes one line of text at a time, which is ideal for large corpora
    sentences = LineSentence(corpus_path)

    # Configure the Word2Vec model
    model = Word2Vec(
        sentences=sentences,       # The processed sentences
        vector_size=vector_size,   # Size of word embeddings
        window=window,             # Context window size
        sg=sg,                     # Use skip-gram (sg=1) or CBOW (sg=0)
        workers=multiprocessing.cpu_count(),  # Use all available CPU cores
        epochs=epochs,             # Number of training epochs
        batch_words=batch_words,   # Number of words processed per batch
        compute_loss=True          # Enable loss tracking
    )

    # Save the model for later use
    model.save("skipgram_word2vec.model")
    print("Model training completed and saved as 'skipgram_word2vec.model'.")
    return model

# Example usage:
train_skipgram_on_large_corpus("TED2013.ar-en.en")


Model training completed and saved as 'skipgram_word2vec.model'.


In [3]:
# Load the saved Word2Vec model
model = Word2Vec.load("skipgram_word2vec.model")

In [4]:
# Print the vocabulary of the trained model.
print("Vocabulary size:", len(model.wv.index_to_key))
print("Sample words in vocabulary:", model.wv.index_to_key[:10])

Vocabulary size: 24879
Sample words in vocabulary: ['the', 'to', 'of', 'a', 'and', 'that', 'in', 'I', 'is', 'you']


In [6]:
from sklearn.decomposition import PCA
import numpy as np  
import tensorflow as tf
import os
from tensorboard.plugins import projector

def visualize_word_embeddings_in_3d(model, max_words=10000):
    """
    Visualizes word embeddings in 3D using TensorBoard.

    Args:
        model (gensim.models.Word2Vec): Trained Word2Vec model containing word embeddings.
        max_words (int): Maximum number of words to visualize (default: 10000)
    """
    # Extract word vectors and corresponding words (limit to max_words for performance)
    words = list(model.wv.index_to_key[:max_words])
    word_vectors = np.array([model.wv[word] for word in words])

    # Reduce dimensionality to 3D using PCA
    pca = PCA(n_components=3)
    word_vectors_3d = pca.fit_transform(word_vectors)
    
    print(f"Original embedding dimension: {word_vectors.shape[1]}")
    print(f"Reduced to 3D. Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")

    # Set up TensorBoard log directory
    log_dir = "logs/word_embeddings"
    os.makedirs(log_dir, exist_ok=True)

    # Save the embeddings as a checkpoint
    checkpoint_path = os.path.join(log_dir, 'embeddings.ckpt')
    
    # Create embeddings variable
    embeddings = tf.Variable(word_vectors_3d, name='word_embeddings')
    
    # Create a checkpoint
    checkpoint = tf.train.Checkpoint(embedding=embeddings)
    checkpoint.save(checkpoint_path)

    # Save metadata (word labels)
    metadata_path = os.path.join(log_dir, 'metadata.tsv')
    with open(metadata_path, 'w', encoding='utf-8') as f:
        for word in words:
            f.write(f"{word}\n")

    # Configure the projector
    config = projector.ProjectorConfig()
    embedding = config.embeddings.add()
    embedding.tensor_name = "embedding/.ATTRIBUTES/VARIABLE_VALUE"
    embedding.metadata_path = 'metadata.tsv'
    
    # Save the projector config
    projector.visualize_embeddings(log_dir, config)

    print(f"\n✓ Word embeddings visualized successfully!")
    print(f"  - Log directory: {log_dir}")
    print(f"  - Number of words: {len(words)}")
    print(f"\nTo view in TensorBoard, run:")
    print(f"  tensorboard --logdir={log_dir}")
    print(f"\nThen open your browser to: http://localhost:6006")

# Usage
visualize_word_embeddings_in_3d(model)

Original embedding dimension: 100
Reduced to 3D. Explained variance ratio: 0.2417

✓ Word embeddings visualized successfully!
  - Log directory: logs/word_embeddings
  - Number of words: 10000

To view in TensorBoard, run:
  tensorboard --logdir=logs/word_embeddings

Then open your browser to: http://localhost:6006


In [7]:
queries_words = [
    "spaghetti", "novel", "cloudy", "jogging", "hue", "studying", "temperature",
    "basketball", "pasta", "cinema", "programming", "literature", "baseball",
    "exhausted", "hobby"
]

corpus_words = [
    "I", "love", "pizza", "I", "adore", "pizza", "She", "enjoys", "a", "good", "book",
    "Reading", "is", "my", "hobby", "The", "cat", "is", "on", "the", "mat", "A", "dog",
    "barks", "loudly", "He", "loves", "playing", "soccer", "We", "are", "going", "to",
    "the", "park", "The", "weather", "is", "beautiful", "today", "My", "favorite", "color",
    "is", "blue", "I", "feel", "tired", "today", "I", "hate", "waiting", "in", "line",
    "She", "hates", "tomatoes", "I", "enjoy", "watching", "movies", "Music", "is", "my",
    "passion", "My", "friend", "is", "funny", "The", "sky", "is", "clear", "tonight",
    "I", "am", "learning", "Python", "Coding", "is", "fun", "This", "is", "a", "great",
    "place", "to", "visit"
]

# For each query word, find the 3 closest words in the corpus and print them along with their similarity scores.
for query_word in queries_words:
    if query_word in model.wv:
        similar_words = model.wv.most_similar(query_word, topn=3)
        print(f"Query Word: '{query_word}'")
        for word, score in similar_words:
            print(f"  Similar Word: '{word}', Similarity Score: {score:.4f}")
    else:
        print(f"Query Word: '{query_word}' not found in vocabulary.")


Query Word: 'spaghetti'
  Similar Word: 'sauce', Similarity Score: 0.8492
  Similar Word: 'Tyrone', Similarity Score: 0.8390
  Similar Word: 'sauce.', Similarity Score: 0.8370
Query Word: 'novel'
  Similar Word: 'fund,', Similarity Score: 0.8240
  Similar Word: 'vocabulary', Similarity Score: 0.8038
  Similar Word: 'Nollywood,', Similarity Score: 0.7973
Query Word: 'cloudy'
  Similar Word: 'faint', Similarity Score: 0.9016
  Similar Word: 'lakes', Similarity Score: 0.8959
  Similar Word: 'microbes,', Similarity Score: 0.8944
Query Word: 'jogging' not found in vocabulary.
Query Word: 'hue' not found in vocabulary.
Query Word: 'studying'
  Similar Word: 'researching', Similarity Score: 0.7384
  Similar Word: 'photographing', Similarity Score: 0.7287
  Similar Word: 'approaching', Similarity Score: 0.7192
Query Word: 'temperature'
  Similar Word: 'depth', Similarity Score: 0.7919
  Similar Word: 'dioxide', Similarity Score: 0.7753
  Similar Word: 'cycles', Similarity Score: 0.7727
Query W

In [9]:
import numpy as np
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

def get_sentence_vector(model, sentence):
    """
    Compute the average vector representation of a sentence.
    
    Args:
        model: Trained Word2Vec model
        sentence: String sentence to convert to vector
        
    Returns:
        numpy array: Average vector of words in the sentence, or None if no words found
    """
    words = simple_preprocess(sentence)
    # Get vectors for words that exist in vocabulary
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    
    if len(word_vectors) == 0:
        return None
    
    # Return the average of all word vectors
    return np.mean(word_vectors, axis=0)

def find_most_similar_sentences(model, query, corpus, topn=3):
    """
    Find the most similar sentences in a corpus to a query sentence.
    
    Args:
        model: Trained Word2Vec model
        query: Query sentence (string)
        corpus: List of corpus sentences (list of strings)
        topn: Number of top similar sentences to return
        
    Returns:
        List of tuples (sentence, similarity_score)
    """
    # Get query vector
    query_vector = get_sentence_vector(model, query)
    
    if query_vector is None:
        return []
    
    # Compute similarity with each corpus sentence
    similarities = []
    for sentence in corpus:
        corpus_vector = get_sentence_vector(model, sentence)
        if corpus_vector is not None:
            # Compute cosine similarity
            similarity = np.dot(query_vector, corpus_vector) / (
                np.linalg.norm(query_vector) * np.linalg.norm(corpus_vector)
            )
            similarities.append((sentence, similarity))
    
    # Sort by similarity (descending) and return top N
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:topn]

# Load your model
model = Word2Vec.load("skipgram_word2vec.model")

# Define queries and corpus
queries = [
    "I like pasta",
    "What is your favorite book?",
    "Is it a nice day?",
    "My dog loves to run",
    "What color do you like?",
    "What are you reading?",
    "How's the weather outside?",
    "I enjoy football",
    "Do you like pizza?",
    "I want to watch a movie",
    "Tell me about Python programming",
    "Do you enjoy books?",
    "Are you a fan of soccer?",
    "I feel so tired today",
    "Can you recommend a good hobby?"
]

corpus = [
    "I love pizza",
    "I adore pizza",
    "She enjoys a good book",
    "Reading is my hobby",
    "The cat is on the mat",
    "A dog barks loudly",
    "He loves playing soccer",
    "We are going to the park",
    "The weather is beautiful today",
    "My favorite color is blue",
    "I feel tired today",
    "I hate waiting in line",
    "She hates tomatoes",
    "I enjoy watching movies",
    "Music is my passion",
    "My friend is funny",
    "The sky is clear tonight",
    "I am learning Python",
    "Coding is fun",
    "This is a great place to visit"
]

# Find similar sentences for each query
print("=" * 80)
print("SENTENCE SIMILARITY RESULTS")
print("=" * 80)

for query in queries:
    similar_sentences = find_most_similar_sentences(model, query, corpus, topn=3)
    print(f"\nQuery: '{query}'")
    if similar_sentences:
        for idx, (sentence, score) in enumerate(similar_sentences, 1):
            print(f"  {idx}. Similar Sentence: '{sentence}'")
            print(f"     Similarity Score: {score:.4f}")
    else:
        print("  No similar sentences found (query words not in vocabulary)")
    print("-" * 80)

SENTENCE SIMILARITY RESULTS

Query: 'I like pasta'
  1. Similar Sentence: 'I love pizza'
     Similarity Score: 0.5376
  2. Similar Sentence: 'I enjoy watching movies'
     Similarity Score: 0.5314
  3. Similar Sentence: 'A dog barks loudly'
     Similarity Score: 0.5146
--------------------------------------------------------------------------------

Query: 'What is your favorite book?'
  1. Similar Sentence: 'Reading is my hobby'
     Similarity Score: 0.8836
  2. Similar Sentence: 'My friend is funny'
     Similarity Score: 0.8257
  3. Similar Sentence: 'Music is my passion'
     Similarity Score: 0.8189
--------------------------------------------------------------------------------

Query: 'Is it a nice day?'
  1. Similar Sentence: 'Coding is fun'
     Similarity Score: 0.8212
  2. Similar Sentence: 'The sky is clear tonight'
     Similarity Score: 0.8211
  3. Similar Sentence: 'This is a great place to visit'
     Similarity Score: 0.8193
-----------------------------------------

In [ ]:
"""
FastText Pretrained Embeddings Loader
Downloads and loads pretrained word vectors from Facebook's FastText
"""

import os
import gzip
import shutil
import requests
from tqdm import tqdm
from gensim.models import KeyedVectors

def download_file(url, filename):
    """
    Download a file with progress bar.
    
    Args:
        url: URL to download from
        filename: Local filename to save to
    """
    print(f"Downloading {filename}...")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(filename, 'wb') as f, tqdm(
        desc=filename,
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))
    
    print(f"✓ Downloaded: {filename}")

def decompress_gz(gz_file, output_file):
    """
    Decompress a .gz file.
    
    Args:
        gz_file: Input .gz file path
        output_file: Output decompressed file path
    """
    print(f"Decompressing {gz_file}...")
    with gzip.open(gz_file, 'rb') as f_in:
        with open(output_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"✓ Decompressed: {output_file}")

def download_fasttext_embeddings(language='en'):
    """
    Download FastText pretrained embeddings for a given language.
    
    Args:
        language: Language code ('en' for English, 'ar' for Arabic)
        
    Returns:
        str: Path to the downloaded .vec file
    """
    base_url = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl"
    
    # File names
    gz_filename = f"cc.{language}.300.vec.gz"
    vec_filename = f"cc.{language}.300.vec"
    
    # Check if already downloaded
    if os.path.exists(vec_filename):
        print(f"✓ {vec_filename} already exists. Skipping download.")
        return vec_filename
    
    # Download if .gz doesn't exist
    if not os.path.exists(gz_filename):
        url = f"{base_url}/{gz_filename}"
        try:
            download_file(url, gz_filename)
        except Exception as e:
            print(f"✗ Error downloading: {e}")
            print(f"\nManual download instructions:")
            print(f"1. Visit: https://fasttext.cc/docs/en/crawl-vectors.html")
            print(f"2. Download: {gz_filename}")
            print(f"3. Place it in the current directory: {os.getcwd()}")
            return None
    
    # Decompress
    if os.path.exists(gz_filename):
        decompress_gz(gz_filename, vec_filename)
        # Optionally remove .gz file to save space
        # os.remove(gz_filename)
        return vec_filename
    
    return None

def load_fasttext_model(language='en', limit=None):
    """
    Load FastText pretrained embeddings.
    
    Args:
        language: Language code ('en' or 'ar')
        limit: Maximum number of word vectors to load (None for all)
        
    Returns:
        KeyedVectors: Loaded word embeddings
    """
    vec_file = f"cc.{language}.300.vec"
    
    if not os.path.exists(vec_file):
        print(f"File {vec_file} not found. Attempting to download...")
        vec_file = download_fasttext_embeddings(language)
        if vec_file is None:
            return None
    
    print(f"\nLoading {vec_file}...")
    print("This may take a few minutes for the full model...")
    
    # Load the model
    model = KeyedVectors.load_word2vec_format(
        vec_file, 
        binary=False, 
        limit=limit  # Set limit to load faster (e.g., 100000 for top 100k words)
    )
    
    print(f"✓ Loaded {len(model)} word vectors")
    return model

# ============================================================================
# USAGE EXAMPLE
# ============================================================================

print("=" * 80)
print("LOADING FASTTEXT PRETRAINED EMBEDDINGS")
print("=" * 80)

# Option 1: Load full models (SLOW - may take 5-10 minutes each)
# en_model = load_fasttext_model('en')
# ar_model = load_fasttext_model('ar')

# Option 2: Load limited models (FASTER - recommended for testing)
print("\n[Loading English model - top 100k words]")
en_model = load_fasttext_model('en', limit=100000)

print("\n[Loading Arabic model - top 100k words]")
ar_model = load_fasttext_model('ar', limit=100000)

# ============================================================================
# TEST THE MODELS
# ============================================================================

if en_model:
    print("\n" + "=" * 80)
    print("ENGLISH MODEL - SIMILARITY QUERIES")
    print("=" * 80)
    
    test_words = ["cat", "king", "python", "computer", "happy"]
    
    for word in test_words:
        if word in en_model:
            print(f"\nMost similar to '{word}':")
            similar = en_model.most_similar(word, topn=5)
            for similar_word, score in similar:
                print(f"  - {similar_word}: {score:.4f}")
        else:
            print(f"\n'{word}' not found in vocabulary")
    
    # Word analogy: king - man + woman = ?
    print("\n" + "-" * 80)
    print("Word Analogy: king - man + woman = ?")
    try:
        result = en_model.most_similar(
            positive=['king', 'woman'], 
            negative=['man'], 
            topn=3
        )
        for word, score in result:
            print(f"  - {word}: {score:.4f}")
    except KeyError as e:
        print(f"  Words not in vocabulary: {e}")

if ar_model:
    print("\n" + "=" * 80)
    print("ARABIC MODEL - SIMILARITY QUERIES")
    print("=" * 80)
    
    # Arabic test words: cat, book, love, computer, happy
    arabic_words = ["قطة", "كتاب", "حب", "كمبيوتر", "سعيد"]
    
    for word in arabic_words:
        if word in ar_model:
            print(f"\nMost similar to '{word}':")
            similar = ar_model.most_similar(word, topn=5)
            for similar_word, score in similar:
                print(f"  - {similar_word}: {score:.4f}")
        else:
            print(f"\n'{word}' not found in vocabulary")

print("\n" + "=" * 80)
print("COMPLETE!")
print("=" * 80)